# Saffron detection — Colab GPU workflow

Run this notebook on a T4 GPU. It is resumable: completed folds are skipped. The evaluation sequence after CV follows the threshold, capture-shortcut, cross-camera, null-test, multi-seed, and prospective locked-test checks.

Runtime estimate: not measured on a completed T4 run. Package installation requires one runtime restart; do not treat smoke-test metrics as results.


In [ ]:
import os, subprocess, sys, json

print('GPU check:')
r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError('No GPU detected. Use Colab T4 or a Linux GPU host. This workflow is not valid on CPU for full training.')
print(r.stdout.strip())

print('GPU available; install the pinned project dependencies in the next setup cell.')


In [ ]:
# Install pinned binary dependencies once. Restart the runtime before importing them.
from pathlib import Path
import subprocess, sys

dependency_marker = Path('/content/.saffron_colab_dependencies_installed')
if dependency_marker.exists():
    import importlib.metadata as metadata
    import numpy as np
    import scipy
    import tensorflow as tf
    expected = {'numpy': '2.2.6', 'scipy': '1.16.3', 'tensorflow': '2.20.0', 'keras': '3.13.0'}
    actual = {name: metadata.version(name) for name in expected}
    print('Verified dependency versions:', actual)
    if actual != expected:
        raise RuntimeError(f'Colab dependency versions differ from pinned versions: {actual}')
    devices = tf.config.list_physical_devices('GPU')
    print('TensorFlow GPUs:', devices)
    if not devices:
        raise RuntimeError('TensorFlow imported, but no GPU is visible. Select a GPU runtime.')
else:
    install_command = [
        sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '--upgrade',
        'numpy==2.2.6', 'scipy==1.16.3', 'pandas==2.2.3',
        'scikit-learn==1.6.1', 'Pillow==12.2.0',
        'tensorflow==2.20.0', 'keras==3.13.0',
    ]
    print('Installing pinned dependencies:', ' '.join(install_command))
    subprocess.run(install_command, check=True)
    dependency_marker.write_text('Installed; restart the Colab runtime before continuing.\n')
    globals()['_SAFFRON_RESTART_REQUIRED'] = True
    print('Dependencies installed. STOP here, choose Runtime > Restart session, then rerun the notebook from the top.')


In [ ]:
# Get or reuse the repo and copy the private dataset archives from Google Drive
if globals().get('_SAFFRON_RESTART_REQUIRED'):
    raise RuntimeError('Restart the Colab runtime now, then rerun from the top; do not run later cells in this kernel.')
from pathlib import Path
import os, subprocess, sys, shutil
REPO_DIR = Path('/content/saffron')
if (REPO_DIR / '.git').is_dir():
    print('Using existing repository at', REPO_DIR)
elif REPO_DIR.exists():
    raise RuntimeError(f'{REPO_DIR} exists but is not a Git checkout. Rename or remove that specific incomplete directory, then rerun this cell.')
else:
    subprocess.run(['git', 'clone', 'https://github.com/ariktadas144/Saffron-Authentication-Detector.git', str(REPO_DIR)], check=True)
%cd /content/saffron

# Put both original dataset ZIPs in Drive/MyDrive/saffron_data.
# If they are not there, this cell checks the Drive root and then offers browser upload.
from google.colab import drive
from google.colab import files

drive.mount('/content/drive')
DRIVE_DATA_DIR = Path('/content/drive/MyDrive/saffron_data')
Path('data').mkdir(exist_ok=True)
archive_files = sorted(Path('data').glob('*.zip'))
if len(archive_files) == 2:
    print('Reusing dataset archives already copied into data/.')
else:
    archive_files = sorted(DRIVE_DATA_DIR.glob('*.zip')) if DRIVE_DATA_DIR.is_dir() else []
if len(archive_files) != 2 and len(sorted(Path('data').glob('*.zip'))) != 2:
    drive_root_archives = sorted(Path('/content/drive/MyDrive').glob('*.zip'))
    if len(drive_root_archives) == 2:
        archive_files = drive_root_archives
        print('Found exactly 2 dataset archives in the My Drive root.')
if len(archive_files) != 2 and len(sorted(Path('data').glob('*.zip'))) != 2:
    print(
        f'Found {len(archive_files)} ZIP file(s) in {DRIVE_DATA_DIR}. '
        'Select the two original dataset ZIP files in the upload dialog. '
        'Alternatively, stop this cell, place them in Google Drive/My Drive/saffron_data, '
        'and rerun it. The GitHub repository does not contain private dataset files.'
    )
    uploaded = files.upload()
    archive_files = sorted(Path.cwd() / name for name in uploaded if name.lower().endswith('.zip'))
if len(archive_files) != 2:
    raise RuntimeError(
        f'Expected exactly 2 original dataset ZIP files, found {len(archive_files)}: '
        f'{[p.name for p in archive_files]}. Upload both original archives or put them in '
        'Google Drive/My Drive/saffron_data, then rerun this cell.'
    )
for archive in archive_files:
    destination = Path('data') / archive.name
    if archive.resolve() != destination.resolve():
        shutil.copy2(archive, destination)
    print(f'Available archive: {destination.name} ({destination.stat().st_size:,} bytes)')

if len(sorted(Path('data').glob('*.zip'))) != 2:
    raise RuntimeError('Two original dataset ZIP files must be available in data/.')
print('Repo and dataset archives ready at', os.getcwd())


In [ ]:
# Prepare dataset, then verify that the manifest points to available images
import os, glob, subprocess, sys
import pandas as pd

if not os.path.exists('data/manifest.csv'):
    zips = sorted(glob.glob('data/*.zip'))
    if len(zips) != 2:
        raise RuntimeError(
            f'Expected 2 dataset archives in data/ but found {len(zips)}: {zips}. '
            'Run the preceding Drive-copy cell and place both original ZIP files in '
            'Google Drive/My Drive/saffron_data.'
        )
    command = [sys.executable, 'scripts/prepare_dataset.py', '--src', *zips, '--data', 'data', '--folds', '5']
    print('Running:', ' '.join(command))
    subprocess.run(command, check=True)
else:
    print('Using existing data/manifest.csv')

manifest = pd.read_csv('data/manifest.csv')
if manifest.empty or not {'path', 'label', 'group', 'fold'}.issubset(manifest.columns):
    raise RuntimeError('data/manifest.csv is empty or missing required columns: path,label,group,fold')
manifest_paths = manifest.path.astype(str).str.replace('\\', os.sep, regex=False)
missing_images = [path for path in manifest_paths if not os.path.isfile(path)]
if missing_images:
    raise FileNotFoundError(
        f'{len(missing_images)} manifest images are missing; first paths: {missing_images[:5]}. '
        'Re-run dataset preparation from the two original ZIP files in Drive.'
    )
print(f'Manifest ready: {len(manifest)} images, {manifest.group.nunique()} sample groups, {manifest.fold.nunique()} folds')

# run_cv.py needs the EXIF/capture table; this also creates the shortcut baselines.
subprocess.run([
    sys.executable, 'scripts/audit_shortcuts.py', '--manifest', 'data/manifest.csv',
    '--audit-out', 'data/audit_exif.csv',
    '--summary-out', 'runs/audit/shortcut_baselines.csv',
], check=True)


In [ ]:
# Apply threshold fix and fair image-level baseline thresholds, then evaluate all cached folds.
from pathlib import Path
import sys, subprocess

source_path = Path('src/wombat_beam.py')
source = source_path.read_text(encoding='utf-8')
source = source.replace(
    'from scipy.special import expit, logsumexp',
    'from scipy.special import expit, logit, logsumexp',
)
if 'def _select_threshold(y, p, eps=1e-6):' not in source:
    start = source.index('def select_group_threshold(y, p, groups):')
    end = source.index('\n\n# =====================================================================================', start)
    source = source[:start] + '''def _select_threshold(y, p, eps=1e-6):
    z = logit(np.clip(np.asarray(p, dtype=float), eps, 1 - eps))
    zs = np.unique(z)
    candidates = np.concatenate(([zs[0] - 1.0], (zs[:-1] + zs[1:]) / 2, [zs[-1] + 1.0]))
    scored = [
        (_macro_f1_fast(y, (z >= threshold).astype(int)), threshold)
        for threshold in candidates
    ]
    best_score = max(score for score, _ in scored)
    return float(expit(min(
        (threshold for score, threshold in scored if score == best_score),
        key=abs,
    )))


def select_group_threshold(y, p, groups, eps=1e-6):
    """Choose a sample-level threshold by validation macro-F1, breaking ties toward 0.5."""
    p_group, y_group = aggregate_by_group(p, y, groups, eps=eps)
    return _select_threshold(y_group, p_group, eps=eps)
''' + source[end:]

old_single = '''        rows[f"single view, {h}"] = (b_single(test, h), 0.5)
        val_probs[f"single view, {h}"] = b_single(val, h)'''
new_single = '''        single_name = f"single view, {h}"
        single_test, single_val = b_single(test, h), b_single(val, h)
        rows[single_name] = (single_test, 0.5)
        val_probs[single_name] = single_val
        tuned_name = f"{single_name} (validation-tuned threshold)"
        rows[tuned_name] = (single_test, _select_threshold(val.y, single_val))
        val_probs[tuned_name] = single_val'''
if 'tuned_name = f"{single_name} (validation-tuned threshold)"' not in source:
    if old_single not in source:
        raise RuntimeError('Could not locate the expected single-head baseline code to patch.')
    source = source.replace(old_single, new_single, 1)

old_tta = '''    rows["TTA mean-logit, hafm"] = (b_tta_mean_logit(test, test.head_names[-1]), 0.5)
    val_probs["TTA mean-logit, hafm"] = b_tta_mean_logit(val, val.head_names[-1])'''
new_tta = '''    tta_name = "TTA mean-logit, hafm"
    tta_test = b_tta_mean_logit(test, test.head_names[-1])
    tta_val = b_tta_mean_logit(val, val.head_names[-1])
    rows[tta_name] = (tta_test, 0.5)
    val_probs[tta_name] = tta_val
    tuned_tta_name = f"{tta_name} (validation-tuned threshold)"
    rows[tuned_tta_name] = (tta_test, _select_threshold(val.y, tta_val))
    val_probs[tuned_tta_name] = tta_val'''
if 'tuned_tta_name = f"{tta_name} (validation-tuned threshold)"' not in source:
    if old_tta not in source:
        raise RuntimeError('Could not locate the expected TTA baseline code to patch.')
    source = source.replace(old_tta, new_tta, 1)
source_path.write_text(source, encoding='utf-8')
print('Threshold selection and validation-tuned single/TTA baselines are ready.')

# No --retrain: cached folds are reused; any missing fold is trained by run_cv.py.
subprocess.run([
    sys.executable, 'scripts/run_cv.py', '--manifest', 'data/manifest.csv',
    '--out', 'runs/cv_repro', '--folds', '0', '1', '2', '3', '4',
    '--img-size', '256', '--batch', '16', '--epochs-warm', '3',
    '--epochs-ft', '25', '--mc-samples', '16', '--seed', '42',
], check=True)
print('CV tables above should include the validation-tuned single-head and TTA rows.')

In [ ]:
# Step 2: exposure counterfactual test; Step 3: capture-matched subset evaluation.
!python scripts/exposure_counterfactual.py --manifest data/manifest.csv --audit data/audit_exif.csv --run runs/cv_repro
!python scripts/matched_subset_eval.py --run runs/cv_repro --manifest data/manifest.csv --audit data/audit_exif.csv

In [ ]:
# Step 4: generate both leave-one-camera-out manifests and evaluate each direction.
!python scripts/make_loco_manifests.py --manifest data/manifest.csv
from pathlib import Path
import subprocess, sys

loco_manifests = sorted(Path('data').glob('manifest_loco_*_to_*.csv'))
if len(loco_manifests) != 2:
    raise RuntimeError(f'Expected 2 generated LOCO manifests, found {len(loco_manifests)}: {loco_manifests}')
for loco_manifest in loco_manifests:
    run_dir = Path('runs') / loco_manifest.stem
    command = [
        sys.executable, 'scripts/run_cv.py', '--manifest', str(loco_manifest),
        '--out', str(run_dir), '--folds', '0', '--img-size', '256',
        '--batch', '16', '--epochs-warm', '3', '--epochs-ft', '25',
        '--mc-samples', '16', '--seed', '42',
    ]
    print('Running:', ' '.join(command))
    subprocess.run(command, check=True)

In [ ]:
# Step 5: frozen-embedding cross-validation, followed by the group-label permutation null.
!python scripts/frozen_embedding_cv.py --manifest data/manifest.csv --audit data/audit_exif.csv --cache runs/audit/frozen_embeddings.npz --out runs/audit/frozen_embedding_cv.csv
!python scripts/permutation_null.py --manifest data/manifest.csv --cache runs/audit/frozen_embeddings.npz --permutations 200 --seed 42 --out runs/audit/frozen_embedding_permutation_null.csv

In [ ]:
# Step 6: two additional seeds in separate output folders, then paired comparison across all 3 runs.
!python scripts/run_cv.py --manifest data/manifest.csv --out runs/cv_seed43 --folds 0 1 2 3 4 --img-size 256 --batch 16 --epochs-warm 3 --epochs-ft 25 --mc-samples 16 --seed 43
!python scripts/run_cv.py --manifest data/manifest.csv --out runs/cv_seed44 --folds 0 1 2 3 4 --img-size 256 --batch 16 --epochs-warm 3 --epochs-ft 25 --mc-samples 16 --seed 44
!python scripts/paired_decoder_comparison.py --runs runs/cv_repro runs/cv_seed43 runs/cv_seed44 --out runs/audit/paired_decoder_comparisons.csv

## Step 7 · Prospective locked test (run only after collecting a new controlled photo set)

The capture protocol should be completed before running this step. The lock script requires at least 3 phone models, at least 30 independent real and 30 fake samples per phone, at least 3 images per sample, and a capture log with the required fields. Never tune on, modify, or evaluate the locked test more than once. Set `PROSPECTIVE_IMAGES` and `CAPTURE_LOG` to the collected data paths below. The report-signing key is prompted at runtime and is not stored in the notebook.

In [ ]:
# Step 7: set these paths only after the prospective controlled capture set is complete.
from pathlib import Path
from getpass import getpass
import os, subprocess, sys

PROSPECTIVE_IMAGES = Path('data/prospective/images')
CAPTURE_LOG = Path('data/prospective/capture_log.csv')
LOCKED_DIR = Path('data/locked_test')
if not PROSPECTIVE_IMAGES.is_dir() or not CAPTURE_LOG.is_file():
    raise FileNotFoundError('Set PROSPECTIVE_IMAGES and CAPTURE_LOG to the completed controlled capture set before running this cell.')
if LOCKED_DIR.exists() and any(LOCKED_DIR.iterdir()):
    raise RuntimeError(f'{LOCKED_DIR} is not empty; do not recreate or re-evaluate a locked test set.')
subprocess.run([
    sys.executable, 'scripts/lock_test_set.py', '--images', str(PROSPECTIVE_IMAGES),
    '--capture-log', str(CAPTURE_LOG), '--out', str(LOCKED_DIR),
], check=True)
os.environ['SAFFRON_REPORT_HMAC_KEY'] = getpass('Enter a private HMAC signing key for the report: ')
if not os.environ['SAFFRON_REPORT_HMAC_KEY']:
    raise ValueError('The HMAC signing key cannot be empty.')
subprocess.run([
    sys.executable, 'scripts/evaluate_locked.py', '--locked-dir', str(LOCKED_DIR),
    '--run', 'runs/cv_repro', '--out', 'runs/locked_test/evaluation.csv',
], check=True)
print('The locked test has been evaluated once. Do not rerun this cell.')

In [ ]:
# Zip final results for download
import os, shutil, zipfile
archive = 'runs_for_download.zip'
if os.path.exists(archive):
    os.remove(archive)
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk('runs'):
        for name in files:
            full = os.path.join(root, name)
            arcname = os.path.relpath(full, '.')
            zf.write(full, arcname)
print('Created', archive)
